# HIT137 Assignment 2 (S1 2026)

**Student Repository:** https://github.com/AveryDoan/Assignment-2---Software-Now

This notebook contains complete solutions for:

- **Question 1:** File encryption/decryption with verification
- **Question 2:** Mathematical expression evaluator using recursive descent parsing

The write-up includes problem analysis, design decisions, and implementation details to align with the marking rubric.

## Group Collaboration Setup

This notebook is structured so the group can work in parallel and merge cleanly.

### Member Roles

| Member | Notebook Area | Main Work |
| --- | --- | --- |
| Max | Question 1 | Encryption logic and main program |
| Alvi | Question 1 | Decryption logic and verification |
| Jane | Question 2 | Tokenizer and parser |
| Avery | Question 2 | Evaluator, file I/O, and output formatting |

### How to Work Together

1. Keep your work inside your assigned section.
2. Use the same function names and formatting conventions.
3. Test your part with the sample files before merging.
4. Review the final notebook together and run the full flow once at the end.

### Merge Checklist

- Question 1 encrypts and decrypts correctly.
- Question 2 parses and evaluates all required expressions.
- Output matches the sample files.
- Explanations and comments are clear and complete.

## Question 1: Problem Solving Analysis

### Max: Encryption + Main Program

- Describe the encryption approach here.
- Explain how `encrypt_char()`, `encrypt_file()`, and `main()` work together.
- Add any notes about user input handling or orchestration.

In [ ]:
# Step 1: Take user input as shift1 and shift2

shift1 = int(input("Enter the first shift value: "))
shift2 = int(input("Enter the second shift value: "))

# Step 2: Build Encryption Function:
# -------------------------------
# Step 2.1: The Encryption function need to read the text from "raw_text.txt".

import pandas as pd

df = pd.read_csv("raw_text.txt")
print(df.head())







# Step 3: Build Decryption Function:



#Step 4: Build Verification Function:

: 

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Core character-level functions
# ─────────────────────────────────────────────────────────────────────────────

def encrypt_char(c: str, shift1: int, shift2: int) -> str:
    """
    Encrypt a single character using the assignment cipher rules.

    Letters are shifted within the 26-letter alphabet (wrapping around
    with modulo 26).  Non-letter characters are returned unchanged.
    """
    if c.islower():
        pos = ord(c) - ord('a')        # 0 = 'a', 25 = 'z'

        if pos <= 12:                  # first half: a–m
            new_pos = (pos + shift1 * shift2) % 26
        else:                          # second half: n–z
            new_pos = (pos - (shift1 + shift2)) % 26

        return chr(ord('a') + new_pos)

    elif c.isupper():
        pos = ord(c) - ord('A')        # 0 = 'A', 25 = 'Z'

        if pos <= 12:                  # first half: A–M
            new_pos = (pos - shift1) % 26
        else:                          # second half: N–Z
            new_pos = (pos + shift2 ** 2) % 26

        return chr(ord('A') + new_pos)

    else:
        return c                       # digits, spaces, symbols – unchanged


def decrypt_char(c: str, shift1: int, shift2: int) -> str:
    """
    Decrypt a single character by reversing the encryption.

    Strategy – pre-image search:
      For each candidate letter in the same case (a–m tried first to match
      the encryption priority), check whether encrypting it gives back the
      cipher character.  Return the first match.

    This approach correctly inverts the cipher for any shift values where
    the encryption is injective (no two letters map to the same ciphertext),
    and gives the best available answer otherwise.
    """
    if c.islower():
        # Try a–m (0–12) before n–z (13–25) to mirror encryption priority
        for pos in list(range(0, 13)) + list(range(13, 26)):
            candidate = chr(ord('a') + pos)
            if encrypt_char(candidate, shift1, shift2) == c:
                return candidate
        return c   # fallback: no match (should not happen with valid input)

    elif c.isupper():
        for pos in list(range(0, 13)) + list(range(13, 26)):
            candidate = chr(ord('A') + pos)
            if encrypt_char(candidate, shift1, shift2) == c:
                return candidate
        return c

    else:
        return c   # non-letter characters are not encrypted, return as-is


# ─────────────────────────────────────────────────────────────────────────────
# File-level functions
# ─────────────────────────────────────────────────────────────────────────────

def encrypt_file(shift1: int, shift2: int) -> None:
    """
    Read raw_text.txt, encrypt every character, write to encrypted_text.txt.
    """
    with open('raw_text.txt', 'r', encoding='utf-8') as f:
        raw_text = f.read()

    encrypted_text = ''.join(encrypt_char(c, shift1, shift2) for c in raw_text)

    with open('encrypted_text.txt', 'w', encoding='utf-8') as f:
        f.write(encrypted_text)

    print("Encryption complete  →  encrypted_text.txt")

    # Show a short preview so the user can see the cipher in action
    preview_raw = raw_text[:60].replace('\n', ' ')
    preview_enc = encrypted_text[:60].replace('\n', ' ')
    print(f"  Original  : {preview_raw}")
    print(f"  Encrypted : {preview_enc}")


def decrypt_file(shift1: int, shift2: int) -> None:
    """
    Read encrypted_text.txt, decrypt every character, write to decrypted_text.txt.
    """
    with open('encrypted_text.txt', 'r', encoding='utf-8') as f:
        encrypted_text = f.read()

    decrypted_text = ''.join(decrypt_char(c, shift1, shift2) for c in encrypted_text)

    with open('decrypted_text.txt', 'w', encoding='utf-8') as f:
        f.write(decrypted_text)

    print("\nDecryption complete  →  decrypted_text.txt")


def verify_decryption() -> bool:
    """
    Compare raw_text.txt with decrypted_text.txt and report whether they match.

    Returns True if the files are identical, False otherwise.
    """
    with open('raw_text.txt', 'r', encoding='utf-8') as f:
        raw_text = f.read()

    with open('decrypted_text.txt', 'r', encoding='utf-8') as f:
        decrypted_text = f.read()

    print()
    if raw_text == decrypted_text:
        print("Verification PASSED  ✓  Decrypted text matches the original.")
        return True
    else:
        print("Verification FAILED  ✗  Decrypted text does NOT match the original.")

        # Report the first mismatching position to help with debugging
        for i, (a, b) in enumerate(zip(raw_text, decrypted_text)):
            if a != b:
                print(f"  First mismatch at position {i}: "
                      f"original={repr(a)},  decrypted={repr(b)}")
                break

        total_errors = sum(1 for a, b in zip(raw_text, decrypted_text) if a != b)
        print(f"  Total mismatched characters: {total_errors}")
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Main program
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    # ── Prompt user for shift values ──────────────────────────────────────
    while True:
        try:
            shift1 = int(input("Enter shift1 (positive integer): "))
            shift2 = int(input("Enter shift2 (positive integer): "))
            if shift1 <= 0 or shift2 <= 0:
                print("Both values must be positive. Please try again.\n")
                continue
            break
        except ValueError:
            print("Invalid input – please enter whole numbers.\n")

    print()

    # ── Step 1: Encrypt ───────────────────────────────────────────────────
    encrypt_file(shift1, shift2)

    # ── Step 2: Decrypt ───────────────────────────────────────────────────
    decrypt_file(shift1, shift2)

    # ── Step 3: Verify ────────────────────────────────────────────────────
    verify_decryption()

    print()


if __name__ == '__main__':
    main()


Encryption complete  →  encrypted_text.txt
  Original  : The quick brown fox jumps over the lazy dog beneath the shad
  Encrypted : Hnk waoiq hxuct lud pasvy ubkx znk rgfe jum hktkgzn znk yngj

Decryption complete  →  decrypted_text.txt

Verification PASSED  ✓  Decrypted text matches the original.



### Alvi: Decryption + Verification

- Describe the decryption approach here.
- Explain how `decrypt_char()`, `decrypt_file()`, and `verify_decryption()` work together.
- Add any notes about checking the decrypted output against the original text.

## Question 2: Problem Solving Analysis

### Jane: Tokenizer + Parser

- Describe the tokenizer design here.
- Explain how `tokenize()`, `format_tokens()`, and `parse()` work.
- Include the grammar rules handled by `parse_expr`, `parse_term`, `parse_unary`, and `parse_primary`.

In [1]:
# 1. TOKENIZER
# The tokenizer or a lexer is the first step, it reads the input string and breaks it into smaller pieces/tokens 
# For example, it will take string "2 + 3 * 4" and turn it into a list that’s easier for the parser to understand.
#   "2 + 3 * 4" will be read as NUM(2), OP(+), NUM(3), OP(*), NUM(4), END

def tokenize(expr: str):
    """
    Converts a string into a list of tokens. Each token is a dictionary with 'type' and 'value' keys
        {'type': ..., 'value': ...}
    Returns None if found an invalid character, otherwise returns the token list ending with an END token
    """
    tokens = []
    i = 0  # current index in the string

    while i < len(expr):
        ch = expr[i]

        # Ignore spaces, spaces are not important, just skip over them
        if ch.isspace():
            i += 1

        # For numbers: keep reading digits to form a full number (for example "123")
        elif ch.isdigit():
            j = i
            while j < len(expr) and expr[j].isdigit():
                j += 1

            tokens.append({'type': 'NUM', 'value': int(expr[i:j])})
            i = j  # move past the number

        # For operators
        elif ch in '+-*/':
            tokens.append({'type': 'OP', 'value': ch})
            i += 1

        # For parentheses
        elif ch == '(':
            tokens.append({'type': 'LPAREN', 'value': '('})
            i += 1

        elif ch == ')':
            tokens.append({'type': 'RPAREN', 'value': ')'})
            i += 1

        # For invalid character
        else:
            return None

    # Add END token to mark the end of input
    tokens.append({'type': 'END', 'value': None})
    return tokens

def format_tokens(tokens: list) -> str:
    """
    Turn tokens into a nice readable string.
    For example:
        [NUM:3] [OP:+] [NUM:5] [END]
    """
    parts = []

    for tok in tokens:
        t = tok['type']

        if t == 'END':
            parts.append('[END]')
        elif t == 'NUM':
            parts.append(f"[NUM:{tok['value']}]")
        elif t == 'OP':
            parts.append(f"[OP:{tok['value']}]")
        elif t == 'LPAREN':
            parts.append('[LPAREN:(]')
        elif t == 'RPAREN':
            parts.append('[RPAREN:)]')

    return ' '.join(parts)

In [2]:
# 2. Recursive descent parser
# Now we take the tokens and go to the expression in detail, we will build a tree that represents the structure of the expression.
# Idea: we follow the rules of math (operator precedence) by splitting the work into different functions.
# Order (low to high priority):
#   expr will be + -
#   term will be * /
#   unary will be - (negatives)
#   primary will be numbers and brackets
#
# this way, * and / get handled before + and - automatically.

def parse(tokens: list):
    """
    Build a tree from the tokens
    Returns:
        (tree, None) if it works
        (None, error_message) if something goes wrong
    """
    pos = [0]  # store position in a list so inner functions can change it

    def current():
        # look at current token (don’t move forward)
        return tokens[pos[0]]

    def consume():
        # take current token and move forward.
        tok = tokens[pos[0]]
        pos[0] += 1
        return tok

    # expr: handles + and -
    def parse_expr():
        # First, get the left side
        left = parse_term()

        # Then keep checking for + or -
        while current()['type'] == 'OP' and current()['value'] in ('+', '-'):
            op = consume()['value']
            right = parse_term()

            # Build a tree node
            left = {
                'type': 'binop',
                'op': op,
                'left': left,
                'right': right
            }
        return left


    # term: handles * and / and implicit multiplication
    def parse_term():
        left = parse_unary()

        while True:
            cur = current()

            # Case 1: normal * or /
            if cur['type'] == 'OP' and cur['value'] in ('*', '/'):
                op = consume()['value']
                right = parse_unary()

                left = {
                    'type': 'binop',
                    'op': op,
                    'left': left,
                    'right': right
                }

            # Case 2: implicit multiplication
            # Example: 2*(3+4) or (2)*(3)
            elif cur['type'] in ('NUM', 'LPAREN'):
                right = parse_unary()
                left = {
                    'type': 'binop',
                    'op': '*',
                    'left': left,
                    'right': right
                }
            else:
                break
        return left

    # unary: negative numbers like -5 or --5
    def parse_unary():
        cur = current()

        # If we see '-', it’s a negative (not subtraction yet)
        if cur['type'] == 'OP' and cur['value'] == '-':
            consume()
            operand = parse_unary()
            return {'type': 'neg', 'operand': operand}

        # '+' is not allowed as unary
        if cur['type'] == 'OP' and cur['value'] == '+':
            raise ValueError("Unary '+' is not supported")

        return parse_primary()


    # primary: numbers and (expressions)
    def parse_primary():
        cur = current()

        # Number
        if cur['type'] == 'NUM':
            consume()
            return {'type': 'num', 'value': cur['value']}

        # Brackets
        if cur['type'] == 'LPAREN':
            consume()
            node = parse_expr()

            # Make sure we close the bracket
            if current()['type'] != 'RPAREN':
                raise ValueError("Expected ')'")

            consume()
            return node

        # Errors
        if cur['type'] == 'END':
            raise ValueError("Unexpected end")
        if cur['type'] == 'RPAREN':
            raise ValueError("Unexpected ')'")
        raise ValueError(f"Unexpected token: {cur['type']}")


    # Start parsing
    try:
        tree = parse_expr()

        # After parsing, we should be at END
        if current()['type'] != 'END':
            raise ValueError("Extra stuff after expression")

        return tree, None

    except ValueError as e:
        return None, str(e)

### Avery: Evaluator + Integration

- Describe the evaluator and file integration here.
- Explain how `evaluate_node()`, `format_tree()`, `format_result_str()`, and `evaluate_file()` work.
- Include notes about file I/O, output formatting, and error handling.

In [3]:
# Avery's part: tree formatter, evaluator, and evaluate_file integration

import os

# 1) TREE FORMATTER
# Converts Jane's parse tree (nested dicts) into prefix notation for output.
# Example: 2 + 3 * 4 -> (+ 2 (* 3 4))a


def format_tree(node: dict) -> str:
    """Render a parse-tree node as a prefix-notation string."""
    t = node['type']

    # Number node
    if t == 'num':
        return str(node['value'])

    # Unary negative node
    if t == 'neg':
        return f"(neg {format_tree(node['operand'])})"

    # Binary operator node
    if t == 'binop':
        left_str = format_tree(node['left'])
        right_str = format_tree(node['right'])
        return f"({node['op']} {left_str} {right_str})"

    # Fallback for unexpected node type
    return 'ERROR'

# 2) EVALUATOR
# Walks the parse tree recursively and returns a float result.


def evaluate_node(node: dict) -> float:
    """
    Recursively evaluate a parse-tree node and return the float result.

    Raises ZeroDivisionError if division by zero is encountered.
    """
    t = node['type']

    # Number node
    if t == 'num':
        return float(node['value'])

    # Unary negative node
    if t == 'neg':
        return -evaluate_node(node['operand'])

    # Binary operator node
    if t == 'binop':
        left = evaluate_node(node['left'])
        right = evaluate_node(node['right'])
        op = node['op']

        if op == '+':
            return left + right
        if op == '-':
            return left - right
        if op == '*':
            return left * right
        if op == '/':
            if right == 0:
                raise ZeroDivisionError("Division by zero")
            return left / right

    raise ValueError(f"Unknown node type: {t}")


# 3) RESULT FORMATTER
# Formats result values for the output file:
# - Whole numbers are shown without decimal places.
# - Non-whole numbers are rounded to 4 decimal places.


def format_result_str(value: float) -> str:
    """
    Format a float result for the output file.

    Whole numbers (e.g. 8.0) -> '8'
    Non-whole numbers -> rounded to 4 decimal places (e.g. '1.2346')
    """
    rounded = round(value, 4)

    if rounded == int(rounded):
        return str(int(rounded))

    return f'{rounded:.4f}'


# 4) MAIN PUBLIC FUNCTION: evaluate_file
# Runs the full pipeline for each input expression:
# 1. tokenize
# 2. parse
# 3. format tree
# 4. evaluate
# 5. format result
# Writes output.txt in the same folder as the input file.


def evaluate_file(input_path: str) -> list:
    """
    Read mathematical expressions from input_path (one per line),
    evaluate each using recursive descent parsing, and write results
    to 'output.txt' in the same directory.

    Returns a list of dicts:
        {
            'input':  str  - the original expression string,
            'tree':   str  - prefix-notation tree string, or 'ERROR',
            'tokens': str  - formatted token string, or 'ERROR',
            'result': float or 'ERROR' - computed value on success
        }
    """
    abs_input = os.path.abspath(input_path)
    output_dir = os.path.dirname(abs_input)
    output_path = os.path.join(output_dir, 'output.txt')

    with open(abs_input, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    results = []
    output_blocks = []

    for line in lines:
        expr = line.rstrip('\n')

        # Step 1: tokenize
        tokens = tokenize(expr)

        if tokens is None:
            entry = {
                'input': expr,
                'tree': 'ERROR',
                'tokens': 'ERROR',
                'result': 'ERROR'
            }
            results.append(entry)
            output_blocks.append(
                f'Input: {expr}\nTree: ERROR\nTokens: ERROR\nResult: ERROR'
            )
            continue

        tokens_str = format_tokens(tokens)

        # Step 2: parse
        tree_node, parse_error = parse(tokens)

        if tree_node is None:
            entry = {
                'input': expr,
                'tree': 'ERROR',
                'tokens': tokens_str,
                'result': 'ERROR'
            }
            results.append(entry)
            output_blocks.append(
                f'Input: {expr}\nTree: ERROR\nTokens: {tokens_str}\nResult: ERROR'
            )
            continue

        tree_str = format_tree(tree_node)

        # Step 3: evaluate
        try:
            value = evaluate_node(tree_node)
            result_disp = format_result_str(value)
            entry = {
                'input': expr,
                'tree': tree_str,
                'tokens': tokens_str,
                'result': value
            }
        except (ZeroDivisionError, ValueError, OverflowError):
            result_disp = 'ERROR'
            entry = {
                'input': expr,
                'tree': tree_str,
                'tokens': tokens_str,
                'result': 'ERROR'
            }

        results.append(entry)
        output_blocks.append(
            f'Input: {expr}\nTree: {tree_str}\nTokens: {tokens_str}\nResult: {result_disp}'
        )

    output_content = '\n\n'.join(output_blocks) + '\n'
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(output_content)

    return results

### Quick validation against sample files